# Heart Disease Risk Prediction: Complete Machine Learning Pipeline

**Course:** PRML  |  **Dataset:** Cardiovascular Disease dataset (Kaggle, 70,000 patient records). We take a **stratified random sample of 5,000 patients** with **12 features** and predict whether the patient has **cardiovascular (heart) disease** (`cardio`: 0 = no, 1 = yes).

| Assignment requirement | Where in this notebook |
|---|---|
| Tabular dataset, **more than 5000 samples**, at least 10 features | Section 2 (5,000 samples, 12 features) |
| EDA | Section 3 |
| Classification, complete ML pipeline | Sections 4, 6 |
| K-Means (elbow curve, best k) and DBSCAN | Section 5 |
| Models with all features | Section 6 |
| Feature selection (scikit-learn techniques) | Section 7 |
| PCA and model development using PCA | Section 8 |
| Stratified K-Fold evaluation | Sections 4, 6 to 9 |
| Ensemble: hard voting and soft voting | Section 9 |
| Final test-set evaluation and conclusions | Sections 10 and 11 |

> **Note on the target:** this large public heart dataset has a **binary** label (disease / no disease). The pipeline code (stratified splits, macro-F1, confusion matrix) is written for any number of classes.

**How to run (VS Code):** install the *Python* and *Jupyter* extensions, then `pip install -U numpy pandas matplotlib seaborn scikit-learn ipykernel`. Select a kernel and click **Run All**. The data is downloaded automatically on first run (internet needed). Offline? Download `cardio_train.csv` from Kaggle (*Cardiovascular Disease dataset*, sulianova) and put it next to this notebook; the notebook will use it.

## 1. Setup: import libraries

In [ ]:
import warnings, time, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.feature_selection import (VarianceThreshold, SelectKBest, f_classif,
                                       mutual_info_classif, RFE, SelectFromModel)
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay,
                             silhouette_score, adjusted_rand_score)

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100
RANDOM_STATE = 42   # fixed seed so results are reproducible

## 2. Load the dataset

The full file has 70,000 patients, far more than needed. Steps: (1) remove physically impossible records (e.g. blood pressure of 16000, height 55 cm), (2) convert age from days to years and add **BMI**, (3) draw a **stratified random sample of 5,000 patients** (both classes keep their original proportion). This satisfies the "more than 5000 samples" condition and keeps training fast.

* **Continuous (6):** `age_years`, `height`, `weight`, `ap_hi` (systolic BP), `ap_lo` (diastolic BP), `bmi`
* **Categorical (6):** `gender`, `cholesterol` (1 normal, 2 above, 3 well above), `gluc` (glucose, same scale), `smoke`, `alco`, `active`
* **Target:** `cardio` (0 = no disease, 1 = disease)

In [ ]:
RAW_URL = ("https://raw.githubusercontent.com/Manash-git/"
           "Kaggle-Cardiovascular-Disease-Prediction-using-Random-Forest/main/datasets/cardio_train.csv")

def load_cardio_raw():
    if os.path.exists("cardio_train.csv"):             # local copy (e.g. downloaded from Kaggle)
        return pd.read_csv("cardio_train.csv", sep=";")
    return pd.read_csv(RAW_URL, sep=";")                # mirror of the Kaggle file

N_SAMPLES = 5000
raw = load_cardio_raw()
print("Raw records:", raw.shape)

valid = (raw["ap_hi"].between(70, 250) & raw["ap_lo"].between(40, 200) & (raw["ap_hi"] > raw["ap_lo"])
         & raw["height"].between(120, 220) & raw["weight"].between(30, 200))
print("Removed impossible records:", int((~valid).sum()))
clean = raw[valid].copy()
clean["age_years"] = clean["age"] / 365.25
clean["bmi"] = clean["weight"] / (clean["height"] / 100) ** 2
clean = clean.drop(columns=["id", "age"])

# stratified random sample of 5000 patients
df, _ = train_test_split(clean, train_size=N_SAMPLES, stratify=clean["cardio"], random_state=RANDOM_STATE)
df = df.reset_index(drop=True)

class_names = {0: "No disease", 1: "Heart disease"}
num_cols = ["age_years", "height", "weight", "ap_hi", "ap_lo", "bmi"]
cat_cols = ["gender", "cholesterol", "gluc", "smoke", "alco", "active"]

X = df[num_cols + cat_cols]
y = df["cardio"]
print("\nSampled dataset shape:", df.shape)
print("Features             :", X.shape[1])
print("Classes              :", y.nunique())

## 3. Exploratory Data Analysis (EDA)

### 3.1 First look: shape, types, missing values, duplicates

In [ ]:
display(df.head())
df.info()

In [ ]:
display(df.describe().T.round(2))
print("Missing values  :", int(df.isnull().sum().sum()))
print("Duplicate rows  :", int(df.duplicated().sum()))

### 3.2 Target distribution (is the data balanced?)

In [ ]:
counts = df["cardio"].map(class_names).value_counts()
summary = counts.to_frame("count")
summary["percent"] = (100 * summary["count"] / len(df)).round(2)
display(summary)
counts.plot(kind="bar", figsize=(6, 4), color=["steelblue", "crimson"])
plt.title("Class distribution"); plt.ylabel("Number of patients")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

### 3.3 Distribution of the continuous features

In [ ]:
df[num_cols].hist(bins=30, figsize=(14, 7), color="steelblue", edgecolor="white")
plt.suptitle("Histograms of continuous features", y=1.02)
plt.tight_layout(); plt.show()

### 3.4 Outliers (IQR rule)

In [ ]:
Q1 = df[num_cols].quantile(0.25); Q3 = df[num_cols].quantile(0.75); IQR = Q3 - Q1
mask = (df[num_cols] < Q1 - 1.5 * IQR) | (df[num_cols] > Q3 + 1.5 * IQR)
display(pd.DataFrame({"outliers": mask.sum(), "percent": (100 * mask.mean()).round(2)}))
plt.figure(figsize=(12, 4))
sns.boxplot(data=(df[num_cols] - df[num_cols].mean()) / df[num_cols].std(), orient="h")
plt.title("Boxplots of standardized continuous features"); plt.tight_layout(); plt.show()

### 3.5 Which features separate the classes?

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
for ax, col in zip(axes, ["age_years", "ap_hi", "weight", "bmi"]):
    sns.boxplot(x=df["cardio"].map(class_names), y=df[col], ax=ax, palette="Set2",
                order=list(class_names.values()))
    ax.set_title(col + " by class"); ax.set_xlabel("")
plt.tight_layout(); plt.show()

plt.figure(figsize=(8, 6))
sns.heatmap(df[num_cols + ["cardio"]].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation (continuous features and target)"); plt.tight_layout(); plt.show()

In [ ]:
# Categorical features: share of patients WITH disease inside every value
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col in zip(axes, ["cholesterol", "gluc", "active", "smoke"]):
    (df.groupby(col)["cardio"].mean() * 100).plot(kind="bar", ax=ax, color="indianred")
    ax.set_title(f"% with disease by {col}"); ax.set_ylabel("% with disease"); ax.set_ylim(0, 100)
    ax.tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()

### 3.6 EDA observations (write these in your report)

1. **Data quality:** the raw file contains impossible values (e.g. blood pressure above 1000, height under 100 cm). They were removed before sampling. After cleaning there are **no missing values**.
2. **Balance:** the two classes are almost equal, so accuracy is meaningful. We still use **stratified** splitting and report **macro-F1**.
3. **Scale differences:** `weight` (kg), `ap_hi` (mmHg) and `bmi` live on different scales. Distance-based methods (KNN, SVM, K-Means, DBSCAN) and PCA need **standardization**.
4. **Strongest signals:** systolic/diastolic blood pressure (`ap_hi`, `ap_lo`), `age_years` and `cholesterol` level separate the classes best (see boxplots, bar plots and correlation matrix).
5. **Weak signals:** `alco`, `smoke`, `gender` and `height` carry little information, good candidates to be dropped by feature selection.
6. Categorical features are one-hot encoded, so the 12 original features become **20 model inputs**.

## 4. Train / test split, preprocessing and evaluation strategy

* **Hold-out test set (20%)**, stratified, untouched until Section 10.
* **Stratified K-Fold (K=5)** on the training set for comparing *all* models.
* **Preprocessing in a `ColumnTransformer`:** numeric = StandardScaler; categorical = OneHotEncoder.
* Everything (preprocessing, feature selection, PCA, model) is inside a **`Pipeline`**, so it is re-fitted on the training part of each fold only (**no data leakage**).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print("Train:", X_train.shape, " Test:", X_test.shape)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
SCORING = {"accuracy": "accuracy", "f1_macro": "f1_macro"}
results, registry = [], {}


def make_pre():
    return ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)])


def make_models():
    return {
        "Logistic Regression": LogisticRegression(max_iter=2000),
        "KNN": KNeighborsClassifier(n_neighbors=15),
        "SVM (RBF)": SVC(random_state=RANDOM_STATE),
        "Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE),
        "Random Forest": RandomForestClassifier(n_estimators=200, min_samples_leaf=3, random_state=RANDOM_STATE),
    }


def run_cv(group, name, estimator):
    t0 = time.time()
    cv = cross_validate(estimator, X_train, y_train, cv=skf, scoring=SCORING, n_jobs=-1)
    results.append({"Group": group, "Model": name,
                    "CV Accuracy": cv["test_accuracy"].mean(), "CV Acc Std": cv["test_accuracy"].std(),
                    "CV Macro-F1": cv["test_f1_macro"].mean(), "Time (s)": time.time() - t0})
    registry[(group, name)] = estimator


def show(prefix):
    t = pd.DataFrame(results)
    t = t[t["Group"].str.startswith(prefix)].sort_values("CV Macro-F1", ascending=False)
    return t.reset_index(drop=True).round(4)


# The preprocessed training matrix (used for clustering, feature selection and PCA plots)
pre = make_pre().fit(X_train)
Xp = pre.transform(X_train)
feat_names = [n.split("__")[1] for n in pre.get_feature_names_out()]
print("Model inputs after preprocessing:", Xp.shape[1])

## 5. Unsupervised learning: K-Means and DBSCAN

Clustering ignores the labels. We cluster the preprocessed training data and compare with the real classes afterwards (Adjusted Rand Index: about 0 = random, 1 = perfect).

### 5.1 K-Means: best k with the elbow curve
We use **inertia** (elbow) and the **silhouette score** as a second opinion. The elbow is found automatically as the point farthest from the line joining the ends of the curve.

In [ ]:
def find_elbow(x, y):
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    xn = (x - x.min()) / (x.max() - x.min()); yn = (y - y.min()) / (y.max() - y.min())
    dx, dy = xn[-1] - xn[0], yn[-1] - yn[0]
    dist = np.abs(dx * (yn - yn[0]) - dy * (xn - xn[0])) / np.hypot(dx, dy)
    return int(np.argmax(dist))


k_values = list(range(2, 11))
inertias, sil_scores = [], []
for k in k_values:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(Xp)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(Xp, km.labels_, sample_size=2000, random_state=RANDOM_STATE))

best_k = k_values[find_elbow(k_values, inertias)]
best_k_sil = k_values[int(np.argmax(sil_scores))]
print("Best k from the elbow curve     :", best_k)
print("Best k from silhouette (check)  :", best_k_sil)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(k_values, inertias, "o-", color="navy")
ax[0].axvline(best_k, color="red", linestyle="--", label=f"elbow, k = {best_k}")
ax[0].set_title("Elbow curve (inertia)"); ax[0].set_xlabel("k"); ax[0].set_ylabel("Inertia"); ax[0].legend()
ax[1].plot(k_values, sil_scores, "o-", color="darkgreen")
ax[1].axvline(best_k_sil, color="red", linestyle="--", label=f"best silhouette, k = {best_k_sil}")
ax[1].set_title("Silhouette score"); ax[1].set_xlabel("k"); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
kmeans = KMeans(n_clusters=best_k, n_init=10, random_state=RANDOM_STATE).fit(Xp)
km_labels = kmeans.labels_
print("K-Means (k =", best_k, ")")
print("  Silhouette score :", round(silhouette_score(Xp, km_labels, sample_size=2000, random_state=RANDOM_STATE), 4))
print("  Adjusted Rand Index vs true classes:", round(adjusted_rand_score(y_train, km_labels), 4))
display(pd.crosstab(km_labels, y_train.map(class_names), rownames=["cluster"]))

pca2 = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(Xp)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].scatter(pca2[:, 0], pca2[:, 1], c=km_labels, cmap="tab10", s=10, alpha=0.7)
ax[0].set_title(f"K-Means clusters (k = {best_k})")
ax[1].scatter(pca2[:, 0], pca2[:, 1], c=y_train.values, cmap="viridis", s=10, alpha=0.7)
ax[1].set_title("True disease category")
for a in ax: a.set_xlabel("PC 1"); a.set_ylabel("PC 2")
plt.tight_layout(); plt.show()

### 5.2 DBSCAN: density-based clustering
`min_samples` = 2 x number of continuous features = 12. `eps` is taken from the knee of the **k-distance plot**. Points in no dense region get label **-1 (noise)**.

In [ ]:
min_samples = 2 * len(num_cols)
nn = NearestNeighbors(n_neighbors=min_samples).fit(Xp)
kdist = np.sort(nn.kneighbors(Xp)[0][:, -1])
eps_auto = float(kdist[find_elbow(np.arange(len(kdist)), kdist)])
print("min_samples =", min_samples, "| eps from k-distance knee =", round(eps_auto, 3))

plt.figure(figsize=(8, 4.5))
plt.plot(kdist, color="purple")
plt.axhline(eps_auto, color="red", linestyle="--", label=f"eps = {eps_auto:.2f}")
plt.title(f"k-distance plot (k = {min_samples})"); plt.xlabel("Points sorted by distance")
plt.ylabel("Distance to k-th neighbour"); plt.legend(); plt.tight_layout(); plt.show()


def dbscan_summary(eps):
    labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(Xp)
    return labels, len(set(labels)) - (1 if -1 in labels else 0), 100 * np.mean(labels == -1)

rows = []
for f in [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]:
    _, n_c, noise = dbscan_summary(eps_auto * f)
    rows.append({"eps": round(eps_auto * f, 3), "clusters": n_c, "noise %": round(noise, 1)})
display(pd.DataFrame(rows))

In [ ]:
db_labels, n_clusters, noise_pct = dbscan_summary(eps_auto)
print(f"DBSCAN (eps = {eps_auto:.3f}, min_samples = {min_samples})")
print("  Clusters found:", n_clusters, "| Noise points:", int(np.sum(db_labels == -1)), f"({noise_pct:.1f}%)")
print("  Adjusted Rand Index vs true classes:", round(adjusted_rand_score(y_train, db_labels), 4))

noise = db_labels == -1
plt.figure(figsize=(7, 5.5))
plt.scatter(pca2[noise, 0], pca2[noise, 1], c="black", s=10, alpha=0.6, label="noise")
plt.scatter(pca2[~noise, 0], pca2[~noise, 1], c=db_labels[~noise], cmap="tab10", s=10, alpha=0.7)
plt.title("DBSCAN clusters (PCA 2D view)"); plt.xlabel("PC 1"); plt.ylabel("PC 2")
plt.legend(); plt.tight_layout(); plt.show()

### 5.3 K-Means vs DBSCAN

| | K-Means | DBSCAN |
|---|---|---|
| Needs number of clusters? | Yes (elbow curve) | No |
| Main parameters | `k` | `eps`, `min_samples` |
| Cluster shape | round, similar size | any shape |
| Noise handling | none (all points assigned) | labels noise as -1 |

A low Adjusted Rand Index means the natural clusters differ from the real heart-disease label, so **supervised** models are needed next.

## 6. Supervised models using **all** features

Five classifiers (Logistic Regression, KNN, SVM, Decision Tree, Random Forest), each inside `Pipeline(preprocessing + model)`, evaluated with 5-fold Stratified CV.

In [ ]:
GROUP_ALL = "1. All features"
for name, model in make_models().items():
    run_cv(GROUP_ALL, name, Pipeline([("pre", make_pre()), ("model", model)]))
display(show(GROUP_ALL))

## 7. Feature selection (scikit-learn techniques)

| Type | Technique | Idea |
|---|---|---|
| Filter | `VarianceThreshold` | drop almost-constant features |
| Filter | `SelectKBest(f_classif)` | ANOVA F-test with the class |
| Filter | `SelectKBest(mutual_info_classif)` | mutual information (non-linear) |
| Wrapper | `RFE` + Logistic Regression | repeatedly remove the weakest feature |
| Embedded | `SelectFromModel` + Random Forest | keep the most important features |

Each keeps **K = 8** of the 20 inputs (VarianceThreshold keeps what passes the threshold).

In [ ]:
from functools import partial
K = 8

def make_selectors():
    return {
        "VarianceThreshold": VarianceThreshold(threshold=0.05),
        f"SelectKBest F-test (k={K})": SelectKBest(f_classif, k=K),
        f"SelectKBest MutualInfo (k={K})": SelectKBest(partial(mutual_info_classif, random_state=RANDOM_STATE), k=K),
        f"RFE LogReg (k={K})": RFE(LogisticRegression(max_iter=2000), n_features_to_select=K),
        f"SelectFromModel RF (top {K})": SelectFromModel(
            RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
            max_features=K, threshold=-np.inf),
    }

selected = {}
for name, sel in make_selectors().items():
    sel.fit(Xp, y_train)
    selected[name] = [f for f, keep in zip(feat_names, sel.get_support()) if keep]
    print(f"{name:35s} keeps {len(selected[name]):2d} of {len(feat_names)} inputs")

sel_matrix = pd.DataFrame({n: [int(f in s) for f in feat_names] for n, s in selected.items()}, index=feat_names)
sel_matrix["votes"] = sel_matrix.sum(axis=1)
top = sel_matrix.sort_values("votes", ascending=False)
plt.figure(figsize=(9, 8))
sns.heatmap(top.drop(columns="votes"), cmap="Greens", cbar=False, linewidths=0.4, linecolor="white")
plt.title("Which inputs does each selector keep? (green = kept)"); plt.tight_layout(); plt.show()
print("Inputs chosen by at least 4 of the 5 methods:", top[top["votes"] >= 4].index.tolist())

### 7.1 Do the selected features hurt or help the models?
A linear model (Logistic Regression) and a tree ensemble (Random Forest) are tested; the selector is inside the pipeline so it is fitted on training folds only.

In [ ]:
for sel_name in make_selectors():
    for model_name in ["Logistic Regression", "Random Forest"]:
        steps = [("pre", make_pre()), ("selector", make_selectors()[sel_name]), ("model", make_models()[model_name])]
        run_cv("2. " + sel_name, model_name, Pipeline(steps))

display(show("2."))
print("Reference: all features")
display(show("1.").set_index("Model").loc[["Logistic Regression", "Random Forest"], ["CV Accuracy", "CV Macro-F1"]])

## 8. PCA (Principal Component Analysis)

PCA builds uncorrelated components ordered by explained variance. The data is standardized first (done by the preprocessor).

### 8.1 Explained variance

In [ ]:
pca_full = PCA(random_state=RANDOM_STATE).fit(Xp)
evr = pca_full.explained_variance_ratio_; cum = np.cumsum(evr)
n_95 = int(np.argmax(cum >= 0.95)) + 1; n_90 = int(np.argmax(cum >= 0.90)) + 1
print(f"Components for 90% variance: {n_90} | for 95% variance: {n_95} (out of {Xp.shape[1]})")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].bar(range(1, len(evr) + 1), evr, color="teal"); ax[0].set_title("Scree plot")
ax[0].set_xlabel("Principal component"); ax[0].set_ylabel("Explained variance ratio")
ax[1].plot(range(1, len(cum) + 1), cum, "o-", markersize=3, color="darkorange")
ax[1].axhline(0.95, color="red", linestyle="--", label="95% variance")
ax[1].axvline(n_95, color="gray", linestyle="--", label=f"{n_95} components")
ax[1].set_title("Cumulative explained variance"); ax[1].set_xlabel("Number of components"); ax[1].legend()
plt.tight_layout(); plt.show()

plt.figure(figsize=(8, 6))
sns.scatterplot(x=pca2[:, 0], y=pca2[:, 1], hue=y_train.map(class_names).values, palette="viridis", s=25, alpha=0.7)
plt.title("Training data on the first two principal components")
plt.xlabel("PC 1"); plt.ylabel("PC 2"); plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout(); plt.show()

### 8.2 Model development using PCA
Pipeline: preprocessing > PCA (95% variance) > classifier, same Stratified 5-fold CV.

In [ ]:
GROUP_PCA = "3. PCA (95% variance)"
for name, model in make_models().items():
    run_cv(GROUP_PCA, name, Pipeline([("pre", make_pre()),
                                      ("pca", PCA(n_components=0.95, random_state=RANDOM_STATE)),
                                      ("model", model)]))
pca_table = show("3.")
display(pca_table)
compare = show("1.").set_index("Model")[["CV Accuracy", "CV Macro-F1"]].join(
    pca_table.set_index("Model")[["CV Accuracy", "CV Macro-F1"]], lsuffix=" (all)", rsuffix=" (PCA)")
print("All features vs PCA:"); display(compare.round(4))

## 9. Ensemble learning: hard voting and soft voting

* **Hard voting:** each model predicts a class, the **majority** wins.
* **Soft voting:** the models' predicted **probabilities are averaged**, the highest average wins.

Base models: Logistic Regression, KNN, SVM (with probabilities), Random Forest.

In [ ]:
def make_voting(kind):
    return VotingClassifier([
        ("lr", LogisticRegression(max_iter=2000)),
        ("knn", KNeighborsClassifier(n_neighbors=7)),
        ("svm", SVC(probability=True, random_state=RANDOM_STATE)),
        ("rf", RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)),
    ], voting=kind)

for kind in ["hard", "soft"]:
    run_cv("4. Ensemble (all features)", f"{kind.capitalize()} voting",
           Pipeline([("pre", make_pre()), ("model", make_voting(kind))]))
for kind in ["hard", "soft"]:
    run_cv("4. Ensemble (PCA)", f"{kind.capitalize()} voting",
           Pipeline([("pre", make_pre()), ("pca", PCA(n_components=0.95, random_state=RANDOM_STATE)),
                     ("model", make_voting(kind))]))
display(show("4."))

## 10. Final comparison and test-set evaluation

### 10.1 All experiments (5-fold Stratified CV on the training set)

In [ ]:
all_results = pd.DataFrame(results).sort_values("CV Macro-F1")
labels = all_results["Group"].str.slice(3) + " | " + all_results["Model"]
palette = dict(zip(sorted(all_results["Group"].str.slice(0, 1).unique()), ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]))
colors = [palette[g[0]] for g in all_results["Group"]]
fig, ax = plt.subplots(figsize=(11, 0.32 * len(all_results) + 2))
ax.barh(labels, all_results["CV Macro-F1"], color=colors)
ax.set_xlabel("CV Macro-F1"); ax.set_title("All experiments ranked by cross-validated macro-F1")
ax.set_xlim(max(0, all_results["CV Macro-F1"].min() - 0.05), max(all_results["CV Macro-F1"].max() + 0.05, 0.6))
plt.tight_layout(); plt.show()
print("Colours: blue = all features, green = feature selection, red = PCA, purple = ensembles")
display(pd.DataFrame(results).sort_values("CV Macro-F1", ascending=False).reset_index(drop=True).round(4))

### 10.2 Best configuration on the untouched test set
The best pipeline (highest CV macro-F1) is re-trained on the full training set and evaluated **once** on the test set.

In [ ]:
best_row = pd.DataFrame(results).sort_values("CV Macro-F1", ascending=False).iloc[0]
print("Best by CV macro-F1:", best_row["Group"], "|", best_row["Model"])
best_model = clone(registry[(best_row["Group"], best_row["Model"])]).fit(X_train, y_train)
y_pred = best_model.predict(X_test)
print(f"\nTest accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Test macro-F1 : {f1_score(y_test, y_pred, average='macro'):.4f}\n")
target_names = [class_names[c] for c in sorted(class_names)]
print(classification_report(y_test, y_pred, target_names=target_names, zero_division=0))

cm = confusion_matrix(y_test, y_pred, labels=sorted(class_names), normalize="true")
fig, ax = plt.subplots(figsize=(7.5, 6))
ConfusionMatrixDisplay(cm, display_labels=target_names).plot(ax=ax, cmap="Blues", xticks_rotation=40,
                                                            values_format=".2f", colorbar=False)
ax.set_title("Normalized confusion matrix (test set)"); plt.tight_layout(); plt.show()

In [ ]:
# Hard vs soft voting on the test set (all features) and a single-model reference
rows = []
for kind in ["hard", "soft"]:
    p = Pipeline([("pre", make_pre()), ("model", make_voting(kind))]).fit(X_train, y_train).predict(X_test)
    rows.append({"Model": f"{kind.capitalize()} voting", "Test accuracy": accuracy_score(y_test, p),
                 "Test macro-F1": f1_score(y_test, p, average="macro")})
p = Pipeline([("pre", make_pre()), ("model", make_models()["Random Forest"])]).fit(X_train, y_train).predict(X_test)
rows.append({"Model": "Single Random Forest (reference)", "Test accuracy": accuracy_score(y_test, p),
             "Test macro-F1": f1_score(y_test, p, average="macro")})
display(pd.DataFrame(rows).round(4))

## 11. Summary and conclusions

Fill in the numbers from your own run:

1. **Dataset:** Cardiovascular Disease (Kaggle), cleaned, stratified sample of 5,000 patients, 12 features (20 after one-hot), 2 balanced classes, no missing values.
2. **Clustering:** K-Means elbow gave k = ___; DBSCAN (eps = ___, min_samples = 12) found ___ clusters and ___ % noise. Both have a low Adjusted Rand Index, so unsupervised clusters do not match the disease label.
3. **All features:** best single model ___ with CV accuracy ___ and macro-F1 ___.
4. **Feature selection:** keeping 8 inputs gave accuracy ___ vs ___ with all 20 inputs.
5. **PCA:** ___ components keep 95% variance; PCA models scored ___ (higher / lower).
6. **Ensembles:** hard voting ___, soft voting ___.
7. **Final test result:** best pipeline reached ___ accuracy and ___ macro-F1.

**Possible improvements:** hyper-parameter tuning (`GridSearchCV`), gradient boosting, more engineered features (pulse pressure = ap_hi - ap_lo), and using more than 5,000 patients.